In [ ]:
# =============================================================================
# 1. INSTALL
# =============================================================================
!pip install langchain langchain-groq langchain-community --quiet

import sentence_transformers, langchain
print('All packages ready.')

All packages ready.


In [2]:
# =============================================================================
# 2. CREDENTIALS
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = secrets.get_secret('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = secrets.get_secret('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = secrets.get_secret('AWS_DEFAULT_REGION')
GROQ_API_KEY = secrets.get_secret('GROQ_API_KEY')

print('Credentials loaded.')

Credentials loaded.


In [3]:
# =============================================================================
# 3. LOAD DATA FROM S3
# =============================================================================
import boto3
import pandas as pd

s3     = boto3.client('s3')
BUCKET = 'brazilian-ecommerce-vs'
TABLES = ['orders', 'customers', 'products', 'reviews', 'payments', 'sellers']
dfs = {}
for table in TABLES:
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=f'{table}.csv')
        dfs[table] = pd.read_csv(obj['Body'])
        print(f'  {table}: {dfs[table].shape[0]:,} rows × {dfs[table].shape[1]} cols')
    except Exception as e:
        print(f'  {table}: SKIPPED ({e})')

print('\nAll tables loaded.')

  orders: 99,441 rows × 12 cols
  customers: 99,441 rows × 5 cols
  products: 32,951 rows × 9 cols
  reviews: 99,224 rows × 8 cols
  payments: 103,886 rows × 5 cols
  sellers: 3,095 rows × 4 cols

All tables loaded.


In [4]:
# =============================================================================
# 4. CONVERT ROWS → LANGCHAIN DOCUMENTS
# =============================================================================
from langchain_core.documents import Document

all_docs = []
for table, df in dfs.items():
    for _, row in df.iterrows():
        fields = [f'{col}: {val}' for col, val in row.items() if val is not None and str(val).strip()]
        text = f'[{table}] ' + ' | '.join(fields)
        all_docs.append(Document(page_content=text, metadata={'table': table}))

print(f'Created {len(all_docs):,} documents from {len(dfs)} tables.')

Created 438,038 documents from 6 tables.


In [5]:
# =============================================================================
# 5. EMBEDDINGS (free, runs on Kaggle T4 GPU, no API key)
# =============================================================================
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print('Embedding model loaded.')

/tmp/ipykernel_984/4288525702.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


In [6]:
# =============================================================================
# 6. VECTOR STORE
# =============================================================================
from langchain_core.vectorstores import InMemoryVectorStore

print(f'Embedding {len(all_docs):,} documents...')
vectorstore = InMemoryVectorStore.from_documents(all_docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 10})
print('Vector store ready.')

Embedding 438,038 documents...
Vector store ready.


In [7]:
# =============================================================================
# 7. LLM — Groq (free tier, ~500 req/day, very fast)
# =============================================================================
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=512
)
print('Groq Llama 3.1 ready.')

Groq Llama 3.1 ready.


In [8]:
# =============================================================================
# 8. SCHEMA + RELATIONSHIPS + UNIQUE VALUES
# =============================================================================
import io

schema_parts = []
for name, df in dfs.items():
    col_info = []
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else 'N/A'
        dtype = df[col].dtype
        # For low-cardinality columns show ALL unique values
        if df[col].nunique() <= 20:
            unique_vals = df[col].dropna().unique().tolist()
            col_info.append(f'    {col} ({dtype}) — unique values: {unique_vals}')
        else:
            col_info.append(f'    {col} ({dtype}) — e.g. {sample}')
    schema_parts.append(f"dfs['{name}'] — {len(df):,} rows:\n" + '\n'.join(col_info))

SCHEMA = '\n\n'.join(schema_parts) + """

TABLE RELATIONSHIPS:
- dfs['orders']  + dfs['customers'] → join on 'customer_id'
- dfs['orders']  + dfs['payments']  → join on 'order_id'
- dfs['orders']  + dfs['reviews']   → join on 'order_id'
- dfs['products'] is standalone — NO link to orders or payments
- dfs['sellers']  is standalone — NO link to orders

IMPORTANT RULES:
- All date columns are strings — ALWAYS convert with pd.to_datetime() before date math
- NEVER join tables not listed above
- Product-level order counts are IMPOSSIBLE without order_items table — say so if asked

EXAMPLE CODE PATTERNS:

# Orders by state:
merged = dfs['orders'].merge(dfs['customers'][['customer_id','customer_state']], on='customer_id')
result = merged.groupby('customer_state').size().reset_index(name='order_count').sort_values('order_count', ascending=False)

# Total revenue:
result = dfs['payments']['payment_value'].sum()

# Revenue by state:
merged = dfs['orders'].merge(dfs['customers'][['customer_id','customer_state']], on='customer_id')
merged = merged.merge(dfs['payments'][['order_id','payment_value']], on='order_id')
result = merged.groupby('customer_state')['payment_value'].sum().sort_values(ascending=False)

# Average order value:
result = dfs['payments'].groupby('order_id')['payment_value'].sum().mean()

# Unique customers:
result = dfs['customers']['customer_unique_id'].nunique()

# Repeating customers:
counts = dfs['orders'].merge(dfs['customers'][['customer_id','customer_unique_id']], on='customer_id')
result = (counts.groupby('customer_unique_id').size() > 1).sum()

# Orders by status:
result = dfs['orders']['order_status'].value_counts()

# Average review score:
result = dfs['reviews']['review_score'].mean()

# Average delivery time in days:
o = dfs['orders'].copy()
o['order_purchase_timestamp']      = pd.to_datetime(o['order_purchase_timestamp'], errors='coerce')
o['order_delivered_customer_date'] = pd.to_datetime(o['order_delivered_customer_date'], errors='coerce')
o = o.dropna(subset=['order_purchase_timestamp','order_delivered_customer_date'])
result = (o['order_delivered_customer_date'] - o['order_purchase_timestamp']).dt.days.mean()

# Late orders:
o = dfs['orders'].copy()
o['order_estimated_delivery_date'] = pd.to_datetime(o['order_estimated_delivery_date'], errors='coerce')
o['order_delivered_customer_date'] = pd.to_datetime(o['order_delivered_customer_date'], errors='coerce')
o = o.dropna(subset=['order_estimated_delivery_date','order_delivered_customer_date'])
result = (o['order_delivered_customer_date'] > o['order_estimated_delivery_date']).sum()

# Top product categories:
result = dfs['products']['product_category_name'].value_counts().head(10)

# Sellers by state:
result = dfs['sellers']['seller_state'].value_counts()

# Revenue by month:
merged = dfs['payments'].merge(dfs['orders'][['order_id','order_purchase_timestamp']], on='order_id')
merged['month'] = pd.to_datetime(merged['order_purchase_timestamp'], errors='coerce').dt.to_period('M')
result = merged.groupby('month')['payment_value'].sum()

# Revenue by year:
merged = dfs['payments'].merge(dfs['orders'][['order_id','order_purchase_timestamp']], on='order_id')
merged['year'] = pd.to_datetime(merged['order_purchase_timestamp'], errors='coerce').dt.year
result = merged.groupby('year')['payment_value'].sum()
"""

In [16]:
# =============================================================================
# 9. ROUTER + EXECUTOR
# =============================================================================
import re
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

router_prompt = ChatPromptTemplate.from_template("""
You are a routing assistant. Given a question about a database, decide if it needs:
- "compute": counts, totals, averages, rankings, grouping, trends, unique values, distributions
- "lookup": finding details about one specific record or ID

When in doubt, choose "compute".
Reply with ONLY one word: compute or lookup

Question: {question}
""")

router_chain = router_prompt | llm | StrOutputParser()

codegen_prompt = ChatPromptTemplate.from_template("""
You are a Python/pandas expert. You have access to a dict `dfs` with these exact tables, columns, dtypes, sample values, relationships and example patterns:

{schema}

Write a Python code snippet that answers the question.
- Use ONLY column names exactly as shown in the schema
- Follow the EXAMPLE CODE PATTERNS above as closely as possible
- Store the final answer in a variable called `result`
- Return ONLY executable Python code, no imports, no markdown, no explanation

Question: {question}
""")

codegen_chain = codegen_prompt | llm | StrOutputParser()

fix_prompt = ChatPromptTemplate.from_template("""
You are a Python/pandas expert. The following code produced an error.

Schema:
{schema}

Original question: {question}

Broken code:
{code}

Error: {error}

Fix the code. Use ONLY column names from the schema. Store result in `result`.
Return ONLY executable Python code, no markdown, no explanation.
""")

fix_chain = fix_prompt | llm | StrOutputParser()

rag_prompt = ChatPromptTemplate.from_template("""
You are a friendly, conversational data analyst assistant.
Answer using ONLY the database records below.
Be chatty and engaging — like a helpful colleague, not a robot.
Keep it concise. Never invent data.

Context:
{context}

Question: {question}
Answer:""")

def format_docs(docs):
    return '\n\n'.join(f'[{i+1}] {d.page_content}' for i, d in enumerate(docs))

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

def execute_code(code):
    local_ns = {'dfs': dfs, 'pd': pd}
    exec(code, {}, local_ns)
    return local_ns.get('result', 'No result variable set.')

def run_pandas(question):
    code = codegen_chain.invoke({'schema': SCHEMA, 'question': question})
    code = re.sub(r'```python|```', '', code).strip()
    try:
        result = execute_code(code)
    except Exception as e:
        print(f'[fixing code error: {e}]')
        fixed_code = fix_chain.invoke({'schema': SCHEMA, 'question': question, 'code': code, 'error': str(e)})
        fixed_code = re.sub(r'```python|```', '', fixed_code).strip()
        try:
            result = execute_code(fixed_code)
        except Exception as e2:
            return "Hmm, I wasn't able to compute that — it might require data I don't have access to. Try rephrasing or ask me something else!"

    format_prompt = f'''Question: {question}

Computed result:
{result}

Answer in a friendly, conversational tone. Be concise but engaging.
If the result is a table or list, present it cleanly.
Don't say "based on the data" or "the result is" — just answer naturally like a helpful colleague.'''

    return llm.invoke(format_prompt).content

def is_exact_lookup(question):
    """Detect any long alphanumeric ID (order_id, customer_id etc.)"""
    return bool(re.search(r'[a-f0-9]{20,}', question.lower()))

def run_exact_lookup(question):
    """Find the ID directly in dataframes — no embedding search."""
    match = re.search(r'([a-f0-9]{20,})', question.lower())
    if not match:
        return "Couldn't find an ID in your question."

    lookup_id = match.group(1)
    found_rows = []

    for table, df in dfs.items():
        for col in df.columns:
            mask = df[col].astype(str).str.lower() == lookup_id
            if mask.any():
                row = df[mask].iloc[0].to_dict()
                fields = '\n'.join(f'  {k}: {v}' for k, v in row.items() if pd.notna(v) and str(v) != 'nan')
                found_rows.append(f"Table '{table}':\n{fields}")

    if not found_rows:
        return f"Hmm, I searched everywhere but couldn't find any record matching `{lookup_id}`. Double-check the ID!"

    all_found = '\n\n'.join(found_rows)
    format_prompt = f'''Question: {question}

Found these records:
{all_found}

Present this information in a friendly, conversational way. 
Answer the question directly using the data above.'''
    return llm.invoke(format_prompt).content

def ask(question):
    # Exact ID lookup — bypass embedding search entirely
    if is_exact_lookup(question):
        return f'[direct lookup]\n{run_exact_lookup(question)}'

    route = router_chain.invoke({'question': question}).strip().lower()
    if 'compute' in route:
        return f'[pandas]\n{run_pandas(question)}'
    else:
        return f'[RAG]\n{rag_chain.invoke(question)}'

print('Pipeline ready.')

Pipeline ready.


In [19]:
# =============================================================================
# 10. CHAT LOOP
# =============================================================================
print("Ask anything about your data. Type 'exit' to quit.\n")

while True:
    question = input('You: ').strip()
    if not question:
        continue
    if question.lower() in ('exit', 'quit'):
        break
    print(f'Bot: {ask(question)}\n')

Ask anything about your data. Type 'exit' to quit.



You:  order status of order id: 26a232627994b0bdd4a4975bedde0e63


Bot: [direct lookup]
Let's take a look at the order status for order ID 26a232627994b0bdd4a4975bedde0e63.

It looks like this order was placed on July 16, 2018, at 2:00 PM. The customer's ID is afdde8786fe503ed8704b7b091d9265b, but we won't go into that right now.

The order was approved on the same day, at 2:15 PM. Now, let's talk about the delivery. The order was delivered by the carrier on July 20, 2018, at 2:41 PM. However, the customer received the order on July 23, 2018, at 9:35 PM. 

As for the estimated delivery date, it was initially set for August 7, 2018. But, since the order was delivered within 7 days, it's not considered late.

Now, let's see what the customer thought of the order. They left a review with a score of 5 out of 5, which is excellent! Unfortunately, we don't have any information about the review comment title or message, but it's clear that the customer was very satisfied.

Finally, let's talk about the payment. The customer paid for the order using a credit 

You:  Total revenue?


Bot: [pandas]
The total revenue is $16,008,872.12.



...

You:  averae of review score


Bot: [pandas]
The average review score is 4.09.



You:  number of review scores with less then 3


Bot: [pandas]
I found the information you're looking for. There are 14575 review scores with less than 3.



You:  exit
